In [1]:
import sys; sys.path.append("../src")
import os
import torch, torch.nn as nn
from data_prep import prepare_data
from models import IDSNet
from sklearn.metrics import classification_report

torch.manual_seed(42)
d = prepare_data("../data/KDDTrain.txt", "../data/KDDTest.txt")

X_train = torch.tensor(d["X_train"]); y_train = torch.tensor(d["y_train"], dtype=torch.long)
X_val   = torch.tensor(d["X_val"]);   y_val   = torch.tensor(d["y_val"], dtype=torch.long)
X_test  = torch.tensor(d["X_test"]);  y_test  = torch.tensor(d["y_test"], dtype=torch.long)
print(X_train.shape, X_val.shape, X_test.shape)

torch.Size([100778, 121]) torch.Size([25195, 121]) torch.Size([22544, 121])


In [2]:
model = IDSNet(X_train.shape[1])
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()

batch_size = 256
for epoch in range(10):
    model.train()
    perm = torch.randperm(len(X_train))
    total = 0
    for i in range(0, len(X_train), batch_size):
        idx = perm[i:i + batch_size]
        xb, yb = X_train[idx], y_train[idx]
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(idx)
    print(f"epoch {epoch+1}: loss={total/len(X_train):.4f}  val_acc={accuracy(model, X_val, y_val):.4f}")

epoch 1: loss=0.0849  val_acc=0.9896
epoch 2: loss=0.0238  val_acc=0.9935
epoch 3: loss=0.0190  val_acc=0.9924
epoch 4: loss=0.0168  val_acc=0.9931
epoch 5: loss=0.0147  val_acc=0.9938
epoch 6: loss=0.0149  val_acc=0.9939
epoch 7: loss=0.0132  val_acc=0.9946
epoch 8: loss=0.0135  val_acc=0.9932
epoch 9: loss=0.0117  val_acc=0.9949
epoch 10: loss=0.0112  val_acc=0.9944


In [3]:
model.eval()
with torch.no_grad():
    for name, X, y in [("val", X_val, y_val), ("test", X_test, y_test)]:
        preds = model(X).argmax(dim=1)
        print("=====", name)
        print(classification_report(y.numpy(), preds.numpy(), target_names=["benign", "attack"], digits=3))

os.makedirs("../saved_models", exist_ok=True)
torch.save(model.state_dict(), "../saved_models/nn_baseline.pt")

===== val
              precision    recall  f1-score   support

      benign      0.996     0.993     0.995     13469
      attack      0.993     0.995     0.994     11726

    accuracy                          0.994     25195
   macro avg      0.994     0.994     0.994     25195
weighted avg      0.994     0.994     0.994     25195

===== test
              precision    recall  f1-score   support

      benign      0.679     0.928     0.784      9711
      attack      0.924     0.667     0.775     12833

    accuracy                          0.780     22544
   macro avg      0.801     0.798     0.779     22544
weighted avg      0.818     0.780     0.779     22544

